# GoogLeNet

## 0. 基本认识 GoogLeNet Overview

GoogLeNet 是 2014 年 ILSVRC 图像分类竞赛中的代表性卷积神经网络，对应论文是 *Going Deeper with Convolutions*。它通常也被称为 Inception v1。

它关注的问题是：在不让参数量和计算量失控的前提下，怎样把 CNN 做得更深、更宽。

核心背景可以概括为：

- AlexNet 证明了深层 CNN 在大规模图像分类中很有效，但全连接层参数很多。
- VGG 通过堆叠小卷积核把网络做得更深，但参数量和计算量仍然很大。
- GoogLeNet 用 Inception 模块把不同尺度的卷积分支并联起来，并用 $1 \times 1$ 卷积控制通道数。
- 原版 GoogLeNet 有 22 个带参数层，但参数量大约只有数百万级，远少于 VGG-16。

简单理解：GoogLeNet 不是单纯把卷积层一层层堆深，而是在每个模块里同时看不同尺度的特征，再把结果拼接起来。

---

## 1. Inception 模块 Inception Module

### 1.1 基本结构

Inception 模块的核心是多分支并联。一个典型 Inception v1 模块可以写成：

```text
Branch 1: 1x1 Conv
Branch 2: 1x1 Conv -> 3x3 Conv
Branch 3: 1x1 Conv -> 5x5 Conv
Branch 4: 3x3 MaxPool -> 1x1 Conv
Output: concatenate along channel dimension
```

四个分支会在同一个输入特征图上同时提取特征，最后在通道维度上拼接。PyTorch 中通常使用：

```python
torch.cat([branch1, branch2, branch3, branch4], dim=1)
```

这里的 `dim=1` 对应 `(B, C, H, W)` 中的通道维度 $C$。

### 1.2 为什么要多分支

不同卷积核看到的范围不同：

| 分支 | 作用 | 直观理解 |
| --- | --- | --- |
| $1 \times 1$ 卷积 | 提取通道组合特征 | 看每个位置上的通道关系 |
| $3 \times 3$ 卷积 | 提取局部空间特征 | 看较小邻域 |
| $5 \times 5$ 卷积 | 提取更大范围特征 | 看更大邻域 |
| 最大池化 | 保留显著响应 | 强化局部最明显的特征 |

这种结构让网络不用提前手动决定“只用 3x3 还是只用 5x5”，而是把多个尺度的结果都交给后续层学习。

### 1.3 $1 \times 1$ 卷积的作用

$1 \times 1$ 卷积在 GoogLeNet 中非常关键，主要有两个作用：

- 调整通道数，常用于降维，减少后续 $3 \times 3$ 或 $5 \times 5$ 卷积的计算量。
- 在每个像素位置上做跨通道线性组合，增加特征表达能力。

以输入通道数 $192$、输出 $32$ 个 $5 \times 5$ 卷积核为例，如果直接做 $5 \times 5$ 卷积，参数量约为：

$$
5 \times 5 \times 192 \times 32 = 153600
$$

如果先用 $1 \times 1$ 卷积把通道降到 $16$，再做 $5 \times 5$ 卷积，参数量约为：

$$
1 \times 1 \times 192 \times 16 + 5 \times 5 \times 16 \times 32 = 15872
$$

这样可以明显降低参数量和计算量。

---

## 2. 网络结构 Network Architecture

### 2.1 主线结构

GoogLeNet 的整体结构可以概括为：

```text
Stem 卷积和池化
-> Inception 3a -> Inception 3b -> MaxPool
-> Inception 4a -> Inception 4b -> Inception 4c -> Inception 4d -> Inception 4e -> MaxPool
-> Inception 5a -> Inception 5b
-> Global Average Pooling -> Dropout -> Linear
```

其中 Stem 部分负责把输入图像先变成基础特征图，后面的 Inception 模块负责持续提取多尺度特征。

### 2.2 原版 GoogLeNet 结构

原版 GoogLeNet 常用输入为 $3 \times 224 \times 224$ RGB 图像，输出 1000 个 ImageNet 类别。

| 顺序 | 模块 | 主要层 | 输出尺寸 |
| --- | --- | --- | --- |
| 1 | Input | RGB 图像 | $3 \times 224 \times 224$ |
| 2 | Conv + Pool | $7 \times 7$ Conv, stride=2 + MaxPool | $64 \times 56 \times 56$ |
| 3 | Conv + Pool | $1 \times 1$ Conv + $3 \times 3$ Conv + MaxPool | $192 \times 28 \times 28$ |
| 4 | Inception 3a | 多分支卷积 | $256 \times 28 \times 28$ |
| 5 | Inception 3b | 多分支卷积 | $480 \times 28 \times 28$ |
| 6 | MaxPool | 降采样 | $480 \times 14 \times 14$ |
| 7 | Inception 4a | 多分支卷积 | $512 \times 14 \times 14$ |
| 8 | Inception 4b | 多分支卷积 | $512 \times 14 \times 14$ |
| 9 | Inception 4c | 多分支卷积 | $512 \times 14 \times 14$ |
| 10 | Inception 4d | 多分支卷积 | $528 \times 14 \times 14$ |
| 11 | Inception 4e | 多分支卷积 | $832 \times 14 \times 14$ |
| 12 | MaxPool | 降采样 | $832 \times 7 \times 7$ |
| 13 | Inception 5a | 多分支卷积 | $832 \times 7 \times 7$ |
| 14 | Inception 5b | 多分支卷积 | $1024 \times 7 \times 7$ |
| 15 | AvgPool | 全局平均池化 | $1024 \times 1 \times 1$ |
| 16 | Linear | 分类层 | $1000$ |

### 2.3 和 VGG 的区别

| 对比项 | VGG | GoogLeNet |
| --- | --- | --- |
| 主要思路 | 串行堆叠 $3 \times 3$ 卷积 | 并联多个尺度的 Inception 分支 |
| 网络形态 | 深而规整 | 深且更宽 |
| 参数控制 | 结构简单但全连接层参数多 | 用 $1 \times 1$ 卷积和全局平均池化减少参数 |
| 分类前结构 | 大型全连接分类器 | Global Average Pooling + Linear |

VGG 的重点是“用统一小卷积核把网络加深”，GoogLeNet 的重点是“在一个模块里同时提取多尺度特征，并控制计算量”。

---

## 3. 尺寸和参数计算 Parameter Calculation

### 3.1 卷积和池化输出尺寸

卷积或池化后的空间尺寸仍然使用下面公式：

$$
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
$$

其中：

- $I$：输入尺寸。
- $P$：padding 大小。
- $K$：卷积核或池化核大小。
- $S$：stride 步幅。
- $O$：输出尺寸。

### 3.2 Stem 部分尺寸变化

第一层卷积通常为 $7 \times 7$，stride=2，padding=3，输入尺寸为 $224 \times 224$：

$$
O = \left\lfloor \frac{224 + 2 \times 3 - 7}{2} \right\rfloor + 1 = 112
$$

所以第一层卷积后空间尺寸变为 $112 \times 112$。随后经过 $3 \times 3$ 最大池化，stride=2，常见 GoogLeNet 实现会使用 `ceil_mode=True` 或等效 padding，使尺寸降为 $56 \times 56$。如果直接用默认 floor 公式且不加 padding，$112$ 会算成 $55$，这一点要和具体代码对齐。

再经过后续卷积和一次最大池化后，进入 Inception 3a 前通常是：

```text
3 x 224 x 224
-> 64 x 112 x 112
-> 64 x 56 x 56
-> 192 x 56 x 56
-> 192 x 28 x 28
```

### 3.3 Inception 输出通道数

Inception 模块的输出通道数等于四个分支输出通道数相加。

以 Inception 3a 为例：

| 分支 | 输出通道数 |
| --- | --- |
| $1 \times 1$ Conv | 64 |
| $1 \times 1$ -> $3 \times 3$ Conv | 128 |
| $1 \times 1$ -> $5 \times 5$ Conv | 32 |
| MaxPool -> $1 \times 1$ Conv | 32 |

所以 Inception 3a 的输出通道数为：

$$
64 + 128 + 32 + 32 = 256
$$

如果四个分支都保持相同的空间尺寸 $28 \times 28$，那么输出就是 $256 \times 28 \times 28$。

### 3.4 全局平均池化

GoogLeNet 在分类前使用 Global Average Pooling，把 $1024 \times 7 \times 7$ 变成 $1024 \times 1 \times 1$：

$$
7 \times 7 \rightarrow 1 \times 1
$$

这样展平后只剩 $1024$ 个特征，再接一个线性层输出类别。相比 VGG 中的大型全连接层，这种设计可以减少大量参数。

---

## 4. 关键思想 Key Ideas

### 4.1 多尺度特征提取

图像中的目标可能有大有小，局部纹理和整体结构都可能重要。Inception 模块同时使用 $1 \times 1$、$3 \times 3$、$5 \times 5$ 和池化分支，相当于让网络在同一层里观察不同感受野。

直观理解：

- 小卷积核更关注细节和局部边缘。
- 大卷积核能看到更大范围的上下文。
- 池化分支保留局部最强响应。
- 拼接后交给下一层，让网络自己选择有用特征。

### 4.2 用 $1 \times 1$ 卷积做瓶颈层

如果直接在高通道特征图上使用 $3 \times 3$ 或 $5 \times 5$ 卷积，计算量会很大。GoogLeNet 先用 $1 \times 1$ 卷积降低通道数，再接大卷积核，这个结构可以理解为瓶颈层 Bottleneck。

它的作用是：

- 降低参数量。
- 降低计算量。
- 增加一次非线性变换。
- 让网络更深、更宽时仍然能训练。

### 4.3 辅助分类器 Auxiliary Classifier

原版 GoogLeNet 在中间层接了两个辅助分类器，通常放在 Inception 4a 和 Inception 4d 后面。

训练时，辅助分类器会额外产生 loss，并和主分类器 loss 一起参与优化。常见理解是：

- 给中间层提供更直接的梯度信号。
- 缓解深层网络训练困难。
- 起到一定正则化作用。

推理时一般只使用最后的主分类器输出，辅助分类器可以关闭或忽略。

### 4.4 用全局平均池化替代大规模全连接层

AlexNet 和 VGG 都有参数很多的全连接层。GoogLeNet 在最后使用全局平均池化，把每个通道压缩成一个数，再接分类层。

好处是：

- 参数量明显减少。
- 降低过拟合风险。
- 输出和类别预测之间关系更直接。

---

## 5. PyTorch 实现思路 PyTorch Implementation

当前 `8.GoogLeNet` 目录里只有 notebook，没有配套 `model.py`、`train.py` 或 `test.py`。所以下面按常见 PyTorch 写法整理，不对应某个本地源码文件。

### 5.1 Inception 模块写法

常见实现会先定义基础卷积块，再定义 Inception 模块：

```python
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))
```

Inception 模块的核心是四个分支并联：

```python
class Inception(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3,
                 ch5x5red, ch5x5, pool_proj):
        super().__init__()
        self.branch1 = BasicConv2d(in_channels, ch1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channels, ch3x3red, kernel_size=1),
            BasicConv2d(ch3x3red, ch3x3, kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channels, ch5x5red, kernel_size=1),
            BasicConv2d(ch5x5red, ch5x5, kernel_size=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv2d(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x):
        outputs = [self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)]
        return torch.cat(outputs, dim=1)
```

注意：原版 GoogLeNet 没有 BatchNorm，很多现代 PyTorch 示例会加入 BatchNorm 来让训练更稳定。看代码时要分清“论文原版”和“实战改写版”。

### 5.2 模型主干

模型主干通常由 Stem、多个 Inception 模块、平均池化和分类层组成：

```text
conv1 -> maxpool
conv2 -> conv3 -> maxpool
inception3a -> inception3b -> maxpool
inception4a -> inception4b -> inception4c -> inception4d -> inception4e -> maxpool
inception5a -> inception5b
avgpool -> dropout -> flatten -> fc
```

如果用于 ImageNet，常见设置是：

```text
输入: 3 x 224 x 224
输出: 1000 类
```

如果改成 FashionMNIST 实战版，可以做如下调整：

```text
输入: 1 x 224 x 224
输出: 10 类
第一层 Conv2d 的 in_channels 从 3 改成 1
最后一层 Linear 的 out_features 从 1000 改成 10
```

### 5.3 训练流程

训练流程和前面的 CNN 类似：

1. 准备数据集和 `DataLoader`。
2. 构建 `GoogLeNet(num_classes=...)` 模型。
3. 选择优化器，例如 Adam 或 SGD。
4. 使用交叉熵损失 `nn.CrossEntropyLoss()`。
5. 如果启用辅助分类器，训练 loss 可以写成主 loss 加辅助 loss。
6. 每个 epoch 统计训练集和验证集的 loss、accuracy。
7. 保存验证集准确率最高的模型参数。

启用辅助分类器时，常见 loss 形式是：

$$
loss = loss_{main} + 0.3 \times loss_{aux1} + 0.3 \times loss_{aux2}
$$

推理或测试时只使用主输出即可。

---

## 6. 注意点 Common Pitfalls

### 6.1 不要把 GoogLeNet 和后续 Inception 版本混淆

GoogLeNet 通常指 Inception v1。后面的 Inception-v2、Inception-v3、Inception-v4 对结构做了不少改进，例如卷积分解、BatchNorm、更复杂的 Inception 模块等。学习当前模型时，先按 Inception v1 理解。

### 6.2 拼接维度不要写错

Inception 四个分支的输出要在通道维度拼接。PyTorch 输入一般是 `(B, C, H, W)`，所以拼接维度是 `dim=1`。

如果写成 `dim=2` 或 `dim=3`，就变成在高度或宽度方向拼接，结构含义完全不同。

### 6.3 各分支空间尺寸必须一致

四个分支最后要 `torch.cat`，因此输出的 $H$ 和 $W$ 必须一致。常见做法是：

- $3 \times 3$ 卷积使用 `padding=1`。
- $5 \times 5$ 卷积使用 `padding=2`。
- 最大池化分支使用 `stride=1, padding=1`。

这样四个分支的空间尺寸都不变，只在通道数上发生变化。

### 6.4 池化尺寸要看实现细节

GoogLeNet 的早期池化经常写成 $3 \times 3$、stride=2，但不同实现可能使用 `ceil_mode=True`、padding 或默认 floor 计算。比如 $112 \times 112$ 经过 $3 \times 3$、stride=2 的池化：

```text
默认 floor、不加 padding: floor((112 - 3) / 2) + 1 = 55
使用 ceil_mode=True: ceil((112 - 3) / 2) + 1 = 56
```

所以看结构表时不要只背尺寸，要和代码里的池化参数一起看。

### 6.5 辅助分类器只在训练时使用

辅助分类器主要用于训练阶段帮助梯度传播。测试和部署时一般只看主分类器输出。使用 PyTorch 官方 `torchvision.models.googlenet` 时，要注意 `aux_logits` 参数和模型返回值形式。

### 6.6 原版和实战版不要混写

| 对比项 | 原版 GoogLeNet | 常见小数据集实战版 |
| --- | --- | --- |
| 数据集 | ImageNet | FashionMNIST / CIFAR 等 |
| 输入通道 | 3 通道 RGB | 1 通道或 3 通道 |
| 输入尺寸 | 常见 $224 \times 224$ | 可能 resize 到 $224 \times 224$，也可能改 Stem 结构 |
| 输出类别 | 1000 类 | 按数据集类别数设置 |
| 辅助分类器 | 训练时使用 | 可以关闭或简化 |

画结构图、算尺寸或写代码时，要先确认自己采用的是原版结构还是为了数据集改过的结构。

---

## 7. 总结 Summary

- GoogLeNet 是 Inception v1 的代表模型，核心论文是 *Going Deeper with Convolutions*。
- 它的主要创新是 Inception 模块：多个卷积和池化分支并联，再按通道拼接。
- $1 \times 1$ 卷积是关键，用来跨通道组合特征，也用来降低通道数和计算量。
- GoogLeNet 用全局平均池化替代大规模全连接层，因此参数量比 VGG 小很多。
- 辅助分类器用于训练阶段帮助梯度传播，推理时通常只使用主分类器。
- 学习 GoogLeNet 时，最重要的是理解多尺度分支、通道拼接、瓶颈层和全局平均池化这几个设计。
- 复现代码时要注意分支输出尺寸一致、`torch.cat` 的维度、辅助分类器返回值，以及原版和实战改写版的输入输出差别。

***********